**BBL WIN PROBABILITY - CHECKPOINT MODELLING**

In [42]:
# What this notebook does:
# - Loads checkpoints CSV
# - Train/test split (time-based)
# - Feature engineering (home-perspective + I1/I2 specific)
# - Separate feature sets for Innings 1 vs Innings 2
# - Innings 1: train on *batting-team win*, map to *home win*
# - Innings 2: deterministic rule for Over 20 + model
# - Robust handling of NaN/inf + median imputation + scaling
# - Overall and per-checkpoint metrics for both innings
#
# Dependencies: numpy, pandas, scikit-learn, matplotlib (optional for plots)

from __future__ import annotations
import numpy as np
import pandas as pd
from pathlib import Path

**Load checkpoints and label**

In [45]:
BASE = Path.cwd()
CHK_PATH = BASE / "data" / "bbl_checkpoints.csv"  

chk = pd.read_csv(CHK_PATH, parse_dates=["date"])

# Ensure binary label 0/1
if chk["home_team_result"].dtype == object:
    chk["home_team_result_bin"] = (chk["home_team_result"].str.lower() == "win").astype(int)
else:
    chk["home_team_result_bin"] = chk["home_team_result"].astype(int)

chk = chk.dropna(subset=["home_team_result_bin"]).copy()
target_col = "home_team_result_bin"


**Adding Team Strength Priors**

In [48]:
# ================================
# Recent Head-to-Head (H2H) priors
# ================================
# We build a match-level table from checkpoints (one row per match),
# then compute a *rolling head-to-head win rate* for the current home team
# vs its current opponent, based only on *past* meetings.

import numpy as np
import pandas as pd

# 1) Derive one row per match with teams and home result
match_cols = ["match_id","date","team1","team2","home_team","home_team_result_bin"]
matches_min = (
    chk.sort_values(["date","match_id"])
       .drop_duplicates(subset=["match_id"], keep="first")[match_cols]
       .copy()
)

# Resolve away team from team1/team2
def infer_away(row):
    if pd.isna(row["home_team"]):
        return np.nan
    if row["home_team"] == row["team1"]:
        return row["team2"]
    elif row["home_team"] == row["team2"]:
        return row["team1"]
    else:
        # if "home_team" is None/unknown for a neutral game, we still pick an "opponent"
        # Prefer team2 if home_team==team1 not matched etc.
        return row["team2"]

matches_min["away_team"] = matches_min.apply(infer_away, axis=1)

# 2) Build a long-form team results table (two rows per match: team+opp, did team win?)
team_rows = []
for r in matches_min.itertuples(index=False):
    # home perspective
    team_rows.append({
        "match_id": r.match_id,
        "date":     r.date,
        "team":     r.home_team,
        "opp":      r.away_team,
        "team_won": int(r.home_team_result_bin),  # 1 if home won
    })
    # away perspective (if opponent known)
    if pd.notna(r.away_team):
        team_rows.append({
            "match_id": r.match_id,
            "date":     r.date,
            "team":     r.away_team,
            "opp":      r.home_team,
            "team_won": int(1 - r.home_team_result_bin),  # if home won, away lost
        })

team_df = pd.DataFrame(team_rows).dropna(subset=["team","opp"]).copy()
team_df = team_df.sort_values(["team","opp","date","match_id"]).reset_index(drop=True)

# 3) Rolling H2H win rate for the *current team vs current opp*, using only prior games
K = 5  # lookback window (last K head-to-heads)
team_df["h2h_winrate_k5"] = (
    team_df.groupby(["team","opp"])["team_won"]
           .apply(lambda s: s.shift().rolling(window=K, min_periods=1).mean())
           .reset_index(level=[0,1], drop=True)
)

# 4) Attach the *home team's* H2H prior for this match (home vs away)
home_h2h = (
    team_df[["match_id","team","opp","h2h_winrate_k5"]]
      .rename(columns={"team":"home_team","opp":"away_team"})
)
matches_min = matches_min.merge(
    home_h2h, on=["match_id","home_team","away_team"], how="left"
)

# 5) Neutral prior for first meeting (no history): 0.50
matches_min["h2h_winrate_k5"] = matches_min["h2h_winrate_k5"].fillna(0.50)

# 6) Bring the feature back to *every checkpoint row* via match_id
chk = chk.merge(
    matches_min[["match_id","h2h_winrate_k5"]],
    on="match_id", how="left"
)

# Sanity peek
chk[["match_id","date","home_team","h2h_winrate_k5"]].head(20)


,match_id,date,home_team,h2h_winrate_k5
0,524915,2011-12-16,Sydney Sixers,0.5
1,524915,2011-12-16,Sydney Sixers,0.5
2,524915,2011-12-16,Sydney Sixers,0.5
3,524915,2011-12-16,Sydney Sixers,0.5
4,524915,2011-12-16,Sydney Sixers,0.5
5,524915,2011-12-16,Sydney Sixers,0.5
6,524915,2011-12-16,Sydney Sixers,0.5
7,524916,2011-12-17,Melbourne Stars,0.5
8,524916,2011-12-17,Melbourne Stars,0.5
9,524916,2011-12-17,Melbourne Stars,0.5


**1) Temporal split (no look-ahead leakage)**

In [20]:
CUTOFF = pd.Timestamp("2019-12-01") 
train = chk[chk["date"] < CUTOFF].copy()
test  = chk[chk["date"] >= CUTOFF].copy()
print("Train/Test shapes:", train.shape, test.shape)

Train/Test shapes: (2131, 27) (2333, 27)


**2) Feature Engineering (shared & robust)**

In [23]:
def add_home_perspective_feats(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Coerce numerics safely
    to_num = [
        "run_rate", "req_run_rate", "target_first_innings", "over_end",
        "cum_runs", "cum_wkts", "wkts_in_hand", "is_chasing", "req_runs",
        "overs_remaining", "balls_remaining", "home_is_batting", "innings"
    ]
    for c in to_num:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    # Safe chase fields
    out["overs_remaining"] = out["overs_remaining"].fillna(0).clip(lower=0)
    out["req_runs"]        = out["req_runs"].fillna(0)
    out["req_run_rate"]    = np.divide(
        out["req_runs"].astype(float),
        out["overs_remaining"].astype(float),
        out=np.zeros_like(out["req_runs"], dtype=float),
        where=(out["overs_remaining"] > 0)
    )

    # Simple linear par for 2nd inns, then home-perspective edges
    out["par_runs"] = np.where(out["is_chasing"] == 1,
                               out["target_first_innings"] * (out["over_end"] / 20.0),
                               0.0)
    out["ahead_of_par"] = np.where(out["is_chasing"] == 1,
                                   out["cum_runs"] - out["par_runs"],
                                   0.0)

    out["rate_gap"] = out["run_rate"].fillna(0) - out["req_run_rate"].fillna(0)
    sign = np.where(out["home_is_batting"] == 1, 1.0, -1.0)
    out["home_rate_edge"] = sign * out["rate_gap"]
    out["home_par_edge"]  = sign * out["ahead_of_par"]

    # Terminal flags (I2)
    out["req_rr_impossible"] = ((out["is_chasing"] == 1) & (out["overs_remaining"] == 0) & (out["req_runs"] > 0)).astype(int)
    out["is_terminal"]       = (out["overs_remaining"] == 0).astype(int)

    # Caps to stabilize
    out["run_rate"]     = out["run_rate"].clip(lower=0, upper=20)
    out["req_run_rate"] = out["req_run_rate"].clip(lower=0, upper=30)
    return out

def add_first_innings_feats(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # I1-only informative but safe to compute for all
    out["proj_total_linear"] = out["cum_runs"] + out["run_rate"] * (20.0 - out["over_end"])
    out["rr_overs_used"]     = np.divide(out["cum_runs"], np.maximum(out["over_end"], 1))
    out["wkts_used"]         = 10.0 - out["wkts_in_hand"]
    out["rr_trend"]          = out["run_rate"] - out["rr_overs_used"]
    out["rr_per_wicket"]     = np.divide(out["run_rate"], np.maximum(out["wkts_used"], 1))
    out["proj_total_linear"] = out["proj_total_linear"].clip(lower=0, upper=300)
    out["rr_per_wicket"]     = out["rr_per_wicket"].clip(lower=0, upper=20)
    return out

def apply_first_innings_nans(df: pd.DataFrame) -> pd.DataFrame:
    """Leave chase-only features as NA for Innings 1; compute safely for 2nd."""
    out = df.copy()
    m_i1 = (out["innings"] == 1)
    chase_cols = [
        "is_chasing","target_first_innings","req_runs","overs_remaining",
        "balls_remaining","req_run_rate","home_rate_edge","home_par_edge",
        "is_terminal","req_rr_impossible","par_runs","ahead_of_par"
    ]
    # upcast to nullable types, then set NA for I1
    for c in chase_cols:
        if c in out.columns:
            if c in ["is_terminal","req_rr_impossible"]:
                out[c] = pd.to_numeric(out[c], errors="coerce").astype("Int64")
            else:
                out[c] = pd.to_numeric(out[c], errors="coerce").astype("Float64")
    if chase_cols:
        exist = [c for c in chase_cols if c in out.columns]
        out.loc[m_i1, exist] = pd.NA
    return out

# Apply engineering
train = add_home_perspective_feats(train)
test  = add_home_perspective_feats(test)
train = add_first_innings_feats(train)
test  = add_first_innings_feats(test)
train = apply_first_innings_nans(train)
test  = apply_first_innings_nans(test)


**3) Separate feature sets for I1 vs I2**

In [26]:
# Innings 1: NO chase context; “home_is_batting” used only for mapping later (not in the model)
INN1_FEATS = [
    "over_end","cum_runs","cum_wkts","run_rate","wkts_in_hand",
    "proj_total_linear","rr_overs_used","wkts_used","rr_trend","rr_per_wicket",
    "home_is_batting","h2h_winrate_k5",  # excluded in training model, used only to map batting->home probs
]
INN1_FEATS_MODEL = [f for f in INN1_FEATS if f != "home_is_batting"]

# Innings 2: chase features + home-perspective
INN2_FEATS = [
    "over_end","cum_runs","cum_wkts","run_rate","wkts_in_hand",
    "is_chasing","target_first_innings","req_runs","overs_remaining",
    "balls_remaining","req_run_rate","home_is_batting",
    "home_rate_edge","home_par_edge","is_terminal","req_rr_impossible","h2h_winrate_k5",
]

# Segment splits
train_i1 = train[train["innings"] == 1].copy()
test_i1  = test[test["innings"] == 1].copy()
train_i2 = train[train["innings"] == 2].copy()
test_i2  = test[test["innings"] == 2].copy()

**4) Utilities: X/y builder, drop-all-NaN, rule@20**

In [29]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score, accuracy_score

def build_xy(df: pd.DataFrame, feats: list[str], ycol: str):
    X = df[feats].astype(float).replace([np.inf, -np.inf], np.nan)
    y = df[ycol].astype(int).to_numpy()
    return X, y

def drop_all_nan_columns(X: pd.DataFrame, feats: list[str]):
    keep = [c for c in feats if c in X.columns and not X[c].isna().all()]
    dropped = [c for c in feats if c not in keep]
    if dropped:
        print("Dropping all-NaN columns:", dropped)
    return X[keep], keep

def apply_terminal_chase_rule(test_df: pd.DataFrame, probs: np.ndarray) -> np.ndarray:
    """
    For Innings 2 rows where the chase is terminal (is_terminal==1), set p(home win)
    deterministically using req_runs and home_is_batting.
    Also covers Over 20 automatically.
    """
    p = probs.copy()
    mask = (test_df["innings"] == 2) & (test_df["is_terminal"].fillna(0).astype(int) == 1)
    if mask.any():
        sl = test_df.loc[mask]
        need = sl["req_runs"].fillna(0).to_numpy()              # >0 => chaser still needs runs (failed)
        hib  = sl["home_is_batting"].fillna(0).to_numpy().astype(int)
        # home wins if (home batting & need<=0) OR (home fielding & need>0)
        p_term = ((hib == 1) & (need <= 0)) | ((hib == 0) & (need > 0))
        p[np.where(mask.to_numpy())[0]] = p_term.astype(float)
    return p

def make_batting_target(df: pd.DataFrame, y_home_col: str = "home_team_result_bin") -> np.ndarray:
    """Batting-team win label for Innings 1; later map back to home win."""
    y_home = df[y_home_col].astype(int).values
    hib = df["home_is_batting"].astype(int).values
    return np.where(hib == 1, y_home, 1 - y_home)

**5) Train & Evaluate — Innings 1 (batting target model)**

In [31]:
# Build training for batting outcome, not home; we’ll map back at inference
X_tr_i1, _ = build_xy(train_i1, INN1_FEATS_MODEL, target_col)  # ignore y_home for training
X_te_i1, y_home_te = build_xy(test_i1,  INN1_FEATS_MODEL, target_col)
y_bat_tr = make_batting_target(train_i1, target_col)

X_tr_i1, INN1_USED = drop_all_nan_columns(X_tr_i1, list(X_tr_i1.columns))
X_te_i1 = X_te_i1[INN1_USED]

model_i1 = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(penalty="l2", C=1.0, solver="lbfgs", max_iter=1000)
)
model_i1.fit(X_tr_i1, y_bat_tr)

# Predict batting-win probabilities then map to home-win using home_is_batting on test
p_bat_i1 = model_i1.predict_proba(X_te_i1)[:, 1]
hib_te   = test_i1.loc[X_te_i1.index, "home_is_batting"].astype(int).values
p_home_i1 = np.where(hib_te == 1, p_bat_i1, 1 - p_bat_i1)
p_home_i1 = np.clip(p_home_i1, 1e-6, 1-1e-6)

met_i1 = {
    "segment": "Innings 1",
    "log_loss": log_loss(y_home_te, p_home_i1),
    "brier": brier_score_loss(y_home_te, p_home_i1),
    "auc": roc_auc_score(y_home_te, p_home_i1) if len(np.unique(y_home_te))>1 else np.nan,
    "acc@0.5": accuracy_score(y_home_te, (p_home_i1 >= 0.5).astype(int)),
}

**6) Train & Evaluate — Innings 2 (home label, with rule)**

In [35]:
X_tr_i2, y_tr_i2 = build_xy(train_i2, INN2_FEATS, target_col)
X_te_i2, y_te_i2 = build_xy(test_i2,  INN2_FEATS, target_col)
X_tr_i2, INN2_USED = drop_all_nan_columns(X_tr_i2, list(X_tr_i2.columns))
X_te_i2 = X_te_i2[INN2_USED]

model_i2 = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(penalty="l2", C=1.0, solver="lbfgs", max_iter=1000)
)
model_i2.fit(X_tr_i2, y_tr_i2)

p_i2 = model_i2.predict_proba(X_te_i2)[:, 1]
p_i2 = apply_terminal_chase_rule(test_i2.loc[X_te_i2.index], p_i2)
p_i2 = np.clip(p_i2, 1e-6, 1-1e-6)

met_i2 = {
    "segment": "Innings 2",
    "log_loss": log_loss(y_te_i2, p_i2),
    "brier": brier_score_loss(y_te_i2, p_i2),
    "auc": roc_auc_score(y_te_i2, p_i2) if len(np.unique(y_te_i2))>1 else np.nan,
    "acc@0.5": accuracy_score(y_te_i2, (p_i2 >= 0.5).astype(int)),
}

print("\nOverall metrics:")
print(pd.DataFrame([met_i1, met_i2]))


Overall metrics:
     segment  log_loss     brier       auc   acc@0.5
0  Innings 1  0.626344  0.219251  0.700129  0.640127
1  Innings 2  0.720393  0.176470  0.821687  0.756732


**7) Per-checkpoint models — Innings 1 (batting target)**

In [38]:
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score, accuracy_score

CHECKPOINTS = [6, 10, 15, 20]

def fit_eval_cp_innings1_batting(train_df, test_df, feats_model, y_home_col):
    models_by_cp, rows = {}, []
    for cp in CHECKPOINTS:
        tr = train_df[(train_df["innings"]==1) & (train_df["over_end"]==cp)].copy()
        te = test_df[(test_df["innings"]==1) & (test_df["over_end"]==cp)].copy()
        if tr.empty or te.empty or te[y_home_col].nunique() < 2:
            continue

        X_tr, _ = build_xy(tr, feats_model, y_home_col)
        X_te, y_home_te = build_xy(te, feats_model, y_home_col)
        y_bat_tr = make_batting_target(tr, y_home_col)

        X_tr, feats_used = drop_all_nan_columns(X_tr, list(X_tr.columns))
        X_te = X_te[feats_used]

        pipe = make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            LogisticRegression(penalty="l2", C=1.0, solver="lbfgs", max_iter=1000)
        )
        pipe.fit(X_tr, y_bat_tr)

        p_bat = pipe.predict_proba(X_te)[:, 1]
        hib_te = te.loc[X_te.index, "home_is_batting"].astype(int).values
        p_home = np.where(hib_te == 1, p_bat, 1 - p_bat)
        p_home = np.clip(p_home, 1e-6, 1-1e-6)

        rows.append({
            "innings": 1,
            "over_end": cp,
            "n_test": len(y_home_te),
            "log_loss": log_loss(y_home_te, p_home),
            "brier": brier_score_loss(y_home_te, p_home),
            "auc": roc_auc_score(y_home_te, p_home),
            "acc@0.5": accuracy_score(y_home_te, (p_home >= 0.5).astype(int)),
        })
        models_by_cp[cp] = {"model": pipe, "feats_used": feats_used}
    return models_by_cp, pd.DataFrame(rows).sort_values("over_end")

models_i1_cp, results_i1_cp = fit_eval_cp_innings1_batting(train, test, INN1_FEATS_MODEL, target_col)
print("\nPer-checkpoint metrics — Innings 1 (batting-target):")
display(results_i1_cp)


Per-checkpoint metrics — Innings 1 (batting-target):


,innings,over_end,n_test,log_loss,brier,auc,acc@0.5
0,1,6,326,0.662282,0.234760,0.647167,0.598160
1,1,10,322,0.653477,0.229398,0.677366,0.621118
2,1,15,312,0.604035,0.209472,0.731354,0.666667
3,1,20,296,0.583221,0.200436,0.755594,0.689189


**8) Per-checkpoint models — Innings 2 (home label + rule)**

In [40]:
def fit_eval_cp_innings2(train_df, test_df, feats, ycol):
    models_by_cp, rows = {}, []
    for cp in CHECKPOINTS:
        tr = train_df[(train_df["innings"]==2) & (train_df["over_end"]==cp)].copy()
        te = test_df[(test_df["innings"]==2) & (test_df["over_end"]==cp)].copy()
        if tr.empty or te.empty or te[ycol].nunique() < 2:
            continue

        X_tr, y_tr = build_xy(tr, feats, ycol)
        X_te, y_te = build_xy(te, feats, ycol)
        X_tr, feats_used = drop_all_nan_columns(X_tr, list(X_tr.columns))
        X_te = X_te[feats_used]

        pipe = make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            LogisticRegression(penalty="l2", C=1.0, solver="lbfgs", max_iter=1000)
        )
        pipe.fit(X_tr, y_tr)
        p = pipe.predict_proba(X_te)[:, 1]
        p = apply_terminal_chase_rule(te.loc[X_te.index], p)
        p = np.clip(p, 1e-6, 1-1e-6)

        rows.append({
            "innings": 2,
            "over_end": cp,
            "n_test": len(y_te),
            "log_loss": log_loss(y_te, p),
            "brier": brier_score_loss(y_te, p),
            "auc": roc_auc_score(y_te, p),
            "acc@0.5": accuracy_score(y_te, (p >= 0.5).astype(int)),
        })
        models_by_cp[cp] = {"model": pipe, "feats_used": feats_used}
    return models_by_cp, pd.DataFrame(rows).sort_values("over_end")

models_i2_cp, results_i2_cp = fit_eval_cp_innings2(train, test, INN2_FEATS, target_col)
print("\nPer-checkpoint metrics — Innings 2:")
display(results_i2_cp)



Per-checkpoint metrics — Innings 2:


,innings,over_end,n_test,log_loss,brier,auc,acc@0.5
0,2,6,318,0.613837,0.204398,0.753570,0.704403
1,2,10,312,0.648888,0.192375,0.787356,0.733974
2,2,15,291,0.561979,0.165702,0.841553,0.762887
3,2,20,156,1.328415,0.096154,0.898649,0.903846
